# ZGLMM147 — Follow-up de compras — divisoes de remessa do pedido — Análise Exploratória

**Tabela:** `dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras`
**Transação SAP:** ZGLMM147
**Colunas:** 24 · **Clustering declarado:** `num_pedido_compra`, `cod_material`, `cod_centro`, `dt_criacao`

---

## Como usar

Aperte **Run All**. Todas as células são **independentes** — cada uma consulta a tabela
diretamente. Não há widget, view temporária nem ordem obrigatória entre elas.

## Alertas de partida _(extraídos dos comentários do `DESCRIBE`)_

| Severidade | Alerta |
|---|---|
| 🔴 ALTA | 🔒 **LGPD — `nm_usuario_criacao`.** O comentário instrui: *"tratar como sensível, validar por contagem ou agregado, **nunca amostrar valor literal** nem usar como literal em WHERE"*. Este notebook **mascara a coluna** na amostra e a analisa apenas por contagem. |
| 🔴 ALTA | **Nunca somar `qt_pedido` nem `vl_liquido` sem deduplicar.** Os comentários avisam que ambos são valores do **item**, não da divisão — repetem-se em todas as divisões do mesmo item. A seção 11 traz os totais **com e sem** deduplicação. |
| 🔴 ALTA | **`qt_a_fornecer` negativo é esperado.** Domínio medido em 2026-09-09 sobre 7.513.595 divisões: saldo zero 68,4%, positivo 22,0% e **negativo 9,6%** (over-delivery pela tolerância de excesso). **Não é erro de carga** (seção 18.2). |
| 🟡 MEDIA | **`ind_entrega_completa` é STRING, não boolean** — domínio `X` e vazio. O prefixo `ind_` não significa boolean nesta plataforma. |
| 🟡 MEDIA | **`nm_grupo_compradores` é a denominação do GRUPO DE COMPRADORES**, não do material. O comentário registra que já foi lido como denominação de material por engano. |
| 🟡 MEDIA | **Tipos de transferência não existem nesta tabela** (UB, ZUB, ZRE). O universo exclui pedidos com centro de saída de estoque (RESWK) preenchido — igual à transação SAP. Ao extrair do SAP, aplicar o mesmo filtro (seção 18.5). |
| 🟡 MEDIA | **`dt_remessa` = 9999-12-31 é válido** (sem prazo definido); `00000000` virou NULL. |
| ⚪ BAIXA | **`nm_fornecedor` tem cobertura de 96,68%** — join LEFT, pode vir nulo legitimamente. |

## Seções

| # | Conteúdo |
|---|---|
| 1 | Metadados |
| 2 | Volumetria |
| 3 | Distribuição por centro — escolha do cenário |
| 4 | Granularidade e chave real |
| 5 | Duplicidade |
| **6** | **Preenchimento de todas as colunas** |
| 7 | Cardinalidade |
| 8 | Domínio das categóricas |
| 10 | Perfil numérico |
| **11** | **Totais para conciliação com o SAP** |
| 12 | Datas |
| 13 | Códigos e zeros à esquerda |
| **14** | **Checksum de linha** |
| 15 | Amostra |
| 16 | Distribuição interna |
| 17 | Freshness |
| **18** | **Análises específicas da transação** |
| 19 | Amostra ampliada |
| 20 | Resumo |

> **Aviso:** contagem de linhas não é evidência de qualidade. Ver seções 4, 5 e 14.


## 1. Metadados

In [0]:
DESCRIBE EXTENDED dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras;

In [0]:
-- Formato, tamanho e particoes
DESCRIBE DETAIL dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras;

In [0]:
-- Ultimas gravacoes
DESCRIBE HISTORY dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras LIMIT 20;

## 2. Volumetria

In [0]:
-- 2. VOLUMETRIA
SELECT COUNT(*)                                                        AS linhas,
       COUNT(DISTINCT CONCAT_WS('|', `num_pedido_compra`, `num_item_pedido_compra`, `num_divisao_remessa`)) AS chaves_completas,
       COUNT(DISTINCT `cod_centro`)                                 AS valores_de_cod_centro,
       COUNT(DISTINCT `num_pedido_compra`)                                 AS num_pedido_compra_distintos
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras;

## 3. Distribuição por centro

Base para escolher o cenário de teste: recorte com volume viável (10 mil a 300 mil linhas)
que contenha os casos-limite identificados nas demais seções.

In [0]:
-- 3. DISTRIBUICAO POR cod_centro
SELECT COALESCE(NULLIF(trim(CAST(`cod_centro` AS STRING)), ''), '(vazio)') AS `cod_centro`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras), 2) AS pct,
       CASE WHEN COUNT(*) BETWEEN 10000 AND 300000
            THEN 'CANDIDATO A CENARIO DE TESTE' ELSE '' END AS sugestao
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras
GROUP BY COALESCE(NULLIF(trim(CAST(`cod_centro` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 50;

## 4. Granularidade e chave real

`linhas ÷ chaves distintas`. Razão maior que 1,00 indica dimensão adicional
multiplicando as linhas.

In [0]:
-- GRANULARIDADE
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras),
g AS (
  SELECT 'num_pedido_compra' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `num_pedido_compra` FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras)
UNION ALL
  SELECT 'num_pedido_compra + num_item_pedido_compra' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `num_pedido_compra`, `num_item_pedido_compra` FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras)
UNION ALL
  SELECT 'num_pedido_compra + num_item_pedido_compra + num_divisao_remessa' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `num_pedido_compra`, `num_item_pedido_compra`, `num_divisao_remessa` FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras)
)
SELECT g.chave, t.total AS linhas, g.distintos,
       ROUND(t.total / g.distintos, 4) AS linhas_por_chave,
       CASE WHEN g.distintos = t.total THEN 'CHAVE UNICA'
            ELSE 'NAO UNICA - ha dimensao adicional' END AS veredito
FROM g CROSS JOIN t
ORDER BY linhas_por_chave;

## 5. Duplicidade

Analisando pela chave `num_pedido_compra + cod_material + cod_centro + dt_criacao`.

**Regra:** linhas idênticas = duplicata real (erro de carga).
Linhas distintas = granularidade adicional legítima.

In [0]:
-- 5. CHAVES DUPLICADAS
SELECT `num_pedido_compra`, `cod_material`, `cod_centro`, `dt_criacao`, COUNT(*) AS qtd
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras
GROUP BY `num_pedido_compra`, `cod_material`, `cod_centro`, `dt_criacao`
HAVING COUNT(*) > 1
ORDER BY qtd DESC
LIMIT 30;

In [0]:
-- O QUE DIFERENCIA AS LINHAS DUPLICADAS
-- Colunas sob LGPD excluidas desta analise: nm_usuario_criacao
WITH dup AS (
  SELECT `num_pedido_compra`, `cod_material`, `cod_centro`, `dt_criacao` FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras GROUP BY `num_pedido_compra`, `cod_material`, `cod_centro`, `dt_criacao` HAVING COUNT(*) > 1
),
d AS (
  SELECT c.* FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras c JOIN dup ON c.`num_pedido_compra` <=> dup.`num_pedido_compra` AND c.`cod_material` <=> dup.`cod_material` AND c.`cod_centro` <=> dup.`cod_centro` AND c.`dt_criacao` <=> dup.`dt_criacao`
),
agg AS (
  SELECT `num_pedido_compra`, `cod_material`, `cod_centro`, `dt_criacao`,
         COUNT(DISTINCT `num_item_pedido_compra`) AS `num_item_pedido_compra`,
         COUNT(DISTINCT `num_divisao_remessa`) AS `num_divisao_remessa`,
         COUNT(DISTINCT `cod_grupo_compradores`) AS `cod_grupo_compradores`,
         COUNT(DISTINCT `nm_grupo_compradores`) AS `nm_grupo_compradores`,
         COUNT(DISTINCT `cod_grupo_mercadoria`) AS `cod_grupo_mercadoria`,
         COUNT(DISTINCT `qt_pedido`) AS `qt_pedido`,
         COUNT(DISTINCT `qt_a_fornecer`) AS `qt_a_fornecer`,
         COUNT(DISTINCT `cod_unidade_medida`) AS `cod_unidade_medida`,
         COUNT(DISTINCT `vl_liquido`) AS `vl_liquido`,
         COUNT(DISTINCT `qt_unidade_preco`) AS `qt_unidade_preco`,
         COUNT(DISTINCT `cod_moeda`) AS `cod_moeda`,
         COUNT(DISTINCT `ind_entrega_completa`) AS `ind_entrega_completa`,
         COUNT(DISTINCT `dt_remessa`) AS `dt_remessa`,
         COUNT(DISTINCT `cod_fornecedor`) AS `cod_fornecedor`,
         COUNT(DISTINCT `nm_fornecedor`) AS `nm_fornecedor`,
         COUNT(DISTINCT `cod_incoterms`) AS `cod_incoterms`,
         COUNT(DISTINCT `nm_incoterms_local`) AS `nm_incoterms_local`,
         COUNT(DISTINCT `cod_tipo_pedido_compra`) AS `cod_tipo_pedido_compra`,
         COUNT(DISTINCT `dateingest`) AS `dateingest`
  FROM d GROUP BY `num_pedido_compra`, `cod_material`, `cod_centro`, `dt_criacao`
)
SELECT coluna, max_valores_distintos,
       CASE WHEN max_valores_distintos > 1 THEN 'VARIA - faz parte da chave real'
            ELSE 'constante' END AS veredito
FROM (
  SELECT stack(19,
    'num_item_pedido_compra', MAX(`num_item_pedido_compra`),
    'num_divisao_remessa', MAX(`num_divisao_remessa`),
    'cod_grupo_compradores', MAX(`cod_grupo_compradores`),
    'nm_grupo_compradores', MAX(`nm_grupo_compradores`),
    'cod_grupo_mercadoria', MAX(`cod_grupo_mercadoria`),
    'qt_pedido', MAX(`qt_pedido`),
    'qt_a_fornecer', MAX(`qt_a_fornecer`),
    'cod_unidade_medida', MAX(`cod_unidade_medida`),
    'vl_liquido', MAX(`vl_liquido`),
    'qt_unidade_preco', MAX(`qt_unidade_preco`),
    'cod_moeda', MAX(`cod_moeda`),
    'ind_entrega_completa', MAX(`ind_entrega_completa`),
    'dt_remessa', MAX(`dt_remessa`),
    'cod_fornecedor', MAX(`cod_fornecedor`),
    'nm_fornecedor', MAX(`nm_fornecedor`),
    'cod_incoterms', MAX(`cod_incoterms`),
    'nm_incoterms_local', MAX(`nm_incoterms_local`),
    'cod_tipo_pedido_compra', MAX(`cod_tipo_pedido_compra`),
    'dateingest', MAX(`dateingest`)
  ) AS (coluna, max_valores_distintos)
  FROM agg
)
ORDER BY max_valores_distintos DESC, coluna;

## 6. Preenchimento de TODAS as colunas

**Seção mais importante.** Detecta coluna nunca carregada.

Em validação anterior, esta análise revelou 7 colunas 100% nulas no Datalake —
uma delas preenchida em 97,9% dos registros do SAP. Este erro **não aparece** em amostragem.

Ordene pela coluna `veredito`: os problemas aparecem primeiro.

In [0]:
-- PREENCHIMENTO DE TODAS AS COLUNAS
-- Para boolean, 'zeros_ou_false' conta os valores FALSE.
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras),
perf AS (
  SELECT stack(24,
    'num_pedido_compra', 'string', COUNT_IF(`num_pedido_compra` IS NULL), COUNT_IF(`num_pedido_compra` IS NOT NULL AND lower(trim(`num_pedido_compra`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_pedido_compra`) RLIKE '^0+([.,]0+)?$'),
    'num_item_pedido_compra', 'string', COUNT_IF(`num_item_pedido_compra` IS NULL), COUNT_IF(`num_item_pedido_compra` IS NOT NULL AND lower(trim(`num_item_pedido_compra`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_item_pedido_compra`) RLIKE '^0+([.,]0+)?$'),
    'num_divisao_remessa', 'string', COUNT_IF(`num_divisao_remessa` IS NULL), COUNT_IF(`num_divisao_remessa` IS NOT NULL AND lower(trim(`num_divisao_remessa`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_divisao_remessa`) RLIKE '^0+([.,]0+)?$'),
    'cod_material', 'string', COUNT_IF(`cod_material` IS NULL), COUNT_IF(`cod_material` IS NOT NULL AND lower(trim(`cod_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_material`) RLIKE '^0+([.,]0+)?$'),
    'cod_centro', 'string', COUNT_IF(`cod_centro` IS NULL), COUNT_IF(`cod_centro` IS NOT NULL AND lower(trim(`cod_centro`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_centro`) RLIKE '^0+([.,]0+)?$'),
    'cod_grupo_compradores', 'string', COUNT_IF(`cod_grupo_compradores` IS NULL), COUNT_IF(`cod_grupo_compradores` IS NOT NULL AND lower(trim(`cod_grupo_compradores`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_grupo_compradores`) RLIKE '^0+([.,]0+)?$'),
    'nm_grupo_compradores', 'string', COUNT_IF(`nm_grupo_compradores` IS NULL), COUNT_IF(`nm_grupo_compradores` IS NOT NULL AND lower(trim(`nm_grupo_compradores`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`nm_grupo_compradores`) RLIKE '^0+([.,]0+)?$'),
    'nm_usuario_criacao', 'string', COUNT_IF(`nm_usuario_criacao` IS NULL), COUNT_IF(`nm_usuario_criacao` IS NOT NULL AND lower(trim(`nm_usuario_criacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`nm_usuario_criacao`) RLIKE '^0+([.,]0+)?$'),
    'cod_grupo_mercadoria', 'string', COUNT_IF(`cod_grupo_mercadoria` IS NULL), COUNT_IF(`cod_grupo_mercadoria` IS NOT NULL AND lower(trim(`cod_grupo_mercadoria`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_grupo_mercadoria`) RLIKE '^0+([.,]0+)?$'),
    'qt_pedido', 'decimal(13,3)', COUNT_IF(`qt_pedido` IS NULL), 0L, COUNT_IF(`qt_pedido` = 0),
    'qt_a_fornecer', 'decimal(14,3)', COUNT_IF(`qt_a_fornecer` IS NULL), 0L, COUNT_IF(`qt_a_fornecer` = 0),
    'cod_unidade_medida', 'string', COUNT_IF(`cod_unidade_medida` IS NULL), COUNT_IF(`cod_unidade_medida` IS NOT NULL AND lower(trim(`cod_unidade_medida`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_unidade_medida`) RLIKE '^0+([.,]0+)?$'),
    'vl_liquido', 'double', COUNT_IF(`vl_liquido` IS NULL), 0L, COUNT_IF(`vl_liquido` = 0),
    'qt_unidade_preco', 'decimal(5,0)', COUNT_IF(`qt_unidade_preco` IS NULL), 0L, COUNT_IF(`qt_unidade_preco` = 0),
    'cod_moeda', 'string', COUNT_IF(`cod_moeda` IS NULL), COUNT_IF(`cod_moeda` IS NOT NULL AND lower(trim(`cod_moeda`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_moeda`) RLIKE '^0+([.,]0+)?$'),
    'ind_entrega_completa', 'string', COUNT_IF(`ind_entrega_completa` IS NULL), COUNT_IF(`ind_entrega_completa` IS NOT NULL AND lower(trim(`ind_entrega_completa`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`ind_entrega_completa`) RLIKE '^0+([.,]0+)?$'),
    'dt_remessa', 'date', COUNT_IF(`dt_remessa` IS NULL), 0L, 0L,
    'dt_criacao', 'date', COUNT_IF(`dt_criacao` IS NULL), 0L, 0L,
    'cod_fornecedor', 'string', COUNT_IF(`cod_fornecedor` IS NULL), COUNT_IF(`cod_fornecedor` IS NOT NULL AND lower(trim(`cod_fornecedor`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_fornecedor`) RLIKE '^0+([.,]0+)?$'),
    'nm_fornecedor', 'string', COUNT_IF(`nm_fornecedor` IS NULL), COUNT_IF(`nm_fornecedor` IS NOT NULL AND lower(trim(`nm_fornecedor`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`nm_fornecedor`) RLIKE '^0+([.,]0+)?$'),
    'cod_incoterms', 'string', COUNT_IF(`cod_incoterms` IS NULL), COUNT_IF(`cod_incoterms` IS NOT NULL AND lower(trim(`cod_incoterms`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_incoterms`) RLIKE '^0+([.,]0+)?$'),
    'nm_incoterms_local', 'string', COUNT_IF(`nm_incoterms_local` IS NULL), COUNT_IF(`nm_incoterms_local` IS NOT NULL AND lower(trim(`nm_incoterms_local`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`nm_incoterms_local`) RLIKE '^0+([.,]0+)?$'),
    'cod_tipo_pedido_compra', 'string', COUNT_IF(`cod_tipo_pedido_compra` IS NULL), COUNT_IF(`cod_tipo_pedido_compra` IS NOT NULL AND lower(trim(`cod_tipo_pedido_compra`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_tipo_pedido_compra`) RLIKE '^0+([.,]0+)?$'),
    'dateingest', 'date', COUNT_IF(`dateingest` IS NULL), 0L, 0L
  ) AS (coluna, tipo, nulos, vazios, zeros_ou_false)
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras
)
SELECT p.coluna, p.tipo, p.nulos, p.vazios, p.zeros_ou_false,
       t.total - p.nulos - p.vazios - p.zeros_ou_false                               AS uteis,
       ROUND(100.0 * (t.total - p.nulos - p.vazios - p.zeros_ou_false) / t.total, 2) AS pct_util,
       CASE WHEN p.nulos = t.total                                               THEN '1. 100% NULO'
            WHEN t.total - p.nulos - p.vazios - p.zeros_ou_false <= 0             THEN '2. SEM VALOR UTIL'
            WHEN (t.total - p.nulos - p.vazios - p.zeros_ou_false) < t.total*0.01 THEN '3. QUASE VAZIO'
            ELSE '9. ok' END                                                      AS veredito
FROM perf p CROSS JOIN t
ORDER BY veredito, pct_util, coluna;

## 7. Cardinalidade

Coluna constante é candidata a valor default de carga.

In [0]:
-- CARDINALIDADE
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras),
card AS (
  SELECT stack(24,
    'num_pedido_compra', 'string', approx_count_distinct(`num_pedido_compra`),
    'num_item_pedido_compra', 'string', approx_count_distinct(`num_item_pedido_compra`),
    'num_divisao_remessa', 'string', approx_count_distinct(`num_divisao_remessa`),
    'cod_material', 'string', approx_count_distinct(`cod_material`),
    'cod_centro', 'string', approx_count_distinct(`cod_centro`),
    'cod_grupo_compradores', 'string', approx_count_distinct(`cod_grupo_compradores`),
    'nm_grupo_compradores', 'string', approx_count_distinct(`nm_grupo_compradores`),
    'nm_usuario_criacao', 'string', approx_count_distinct(`nm_usuario_criacao`),
    'cod_grupo_mercadoria', 'string', approx_count_distinct(`cod_grupo_mercadoria`),
    'qt_pedido', 'decimal(13,3)', approx_count_distinct(`qt_pedido`),
    'qt_a_fornecer', 'decimal(14,3)', approx_count_distinct(`qt_a_fornecer`),
    'cod_unidade_medida', 'string', approx_count_distinct(`cod_unidade_medida`),
    'vl_liquido', 'double', approx_count_distinct(`vl_liquido`),
    'qt_unidade_preco', 'decimal(5,0)', approx_count_distinct(`qt_unidade_preco`),
    'cod_moeda', 'string', approx_count_distinct(`cod_moeda`),
    'ind_entrega_completa', 'string', approx_count_distinct(`ind_entrega_completa`),
    'dt_remessa', 'date', approx_count_distinct(`dt_remessa`),
    'dt_criacao', 'date', approx_count_distinct(`dt_criacao`),
    'cod_fornecedor', 'string', approx_count_distinct(`cod_fornecedor`),
    'nm_fornecedor', 'string', approx_count_distinct(`nm_fornecedor`),
    'cod_incoterms', 'string', approx_count_distinct(`cod_incoterms`),
    'nm_incoterms_local', 'string', approx_count_distinct(`nm_incoterms_local`),
    'cod_tipo_pedido_compra', 'string', approx_count_distinct(`cod_tipo_pedido_compra`),
    'dateingest', 'date', approx_count_distinct(`dateingest`)
  ) AS (coluna, tipo, distintos)
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras
)
SELECT c.coluna, c.tipo, c.distintos,
       ROUND(100.0 * c.distintos / t.total, 4) AS pct_distintos,
       CASE WHEN c.distintos <= 1             THEN '1. CONSTANTE'
            WHEN c.distintos <= 3             THEN '2. cardinalidade muito baixa'
            WHEN c.distintos > t.total * 0.95 THEN '3. candidata a identificador'
            ELSE '9. normal' END AS classificacao
FROM card c CROSS JOIN t
ORDER BY c.distintos;

## 8. Domínio das colunas categóricas

Top 8 valores de cada uma.

> 🔒 **Colunas sob LGPD excluídas desta seção:** `nm_usuario_criacao`. São analisadas apenas por contagem na seção 18.

In [0]:
-- DOMINIO DAS CATEGORICAS (top 8 de cada)
(SELECT 'cod_grupo_compradores' AS coluna, CAST(`cod_grupo_compradores` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras GROUP BY `cod_grupo_compradores` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_grupo_mercadoria' AS coluna, CAST(`cod_grupo_mercadoria` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras GROUP BY `cod_grupo_mercadoria` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_unidade_medida' AS coluna, CAST(`cod_unidade_medida` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras GROUP BY `cod_unidade_medida` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_moeda' AS coluna, CAST(`cod_moeda` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras GROUP BY `cod_moeda` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'ind_entrega_completa' AS coluna, CAST(`ind_entrega_completa` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras GROUP BY `ind_entrega_completa` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_incoterms' AS coluna, CAST(`cod_incoterms` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras GROUP BY `cod_incoterms` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_tipo_pedido_compra' AS coluna, CAST(`cod_tipo_pedido_compra` AS STRING) AS valor, COUNT(*) AS qtd
   FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras GROUP BY `cod_tipo_pedido_compra` ORDER BY qtd DESC LIMIT 8)
ORDER BY coluna, qtd DESC;

## 10. Perfil dos campos numéricos

Entre as colunas analisadas, **1 são `double`** — exigem tolerância de 0,005 na comparação com o SAP.

In [0]:
-- PERFIL NUMERICO
SELECT * FROM (
  SELECT stack(4,
    'qt_pedido', 'decimal(13,3)', COUNT(`qt_pedido`), CAST(MIN(`qt_pedido`) AS DOUBLE), CAST(MAX(`qt_pedido`) AS DOUBLE), CAST(AVG(`qt_pedido`) AS DOUBLE), CAST(percentile_approx(`qt_pedido`, 0.5) AS DOUBLE), COUNT_IF(`qt_pedido` < 0), COUNT_IF(`qt_pedido` = 0),
    'qt_a_fornecer', 'decimal(14,3)', COUNT(`qt_a_fornecer`), CAST(MIN(`qt_a_fornecer`) AS DOUBLE), CAST(MAX(`qt_a_fornecer`) AS DOUBLE), CAST(AVG(`qt_a_fornecer`) AS DOUBLE), CAST(percentile_approx(`qt_a_fornecer`, 0.5) AS DOUBLE), COUNT_IF(`qt_a_fornecer` < 0), COUNT_IF(`qt_a_fornecer` = 0),
    'vl_liquido', 'double', COUNT(`vl_liquido`), CAST(MIN(`vl_liquido`) AS DOUBLE), CAST(MAX(`vl_liquido`) AS DOUBLE), CAST(AVG(`vl_liquido`) AS DOUBLE), CAST(percentile_approx(`vl_liquido`, 0.5) AS DOUBLE), COUNT_IF(`vl_liquido` < 0), COUNT_IF(`vl_liquido` = 0),
    'qt_unidade_preco', 'decimal(5,0)', COUNT(`qt_unidade_preco`), CAST(MIN(`qt_unidade_preco`) AS DOUBLE), CAST(MAX(`qt_unidade_preco`) AS DOUBLE), CAST(AVG(`qt_unidade_preco`) AS DOUBLE), CAST(percentile_approx(`qt_unidade_preco`, 0.5) AS DOUBLE), COUNT_IF(`qt_unidade_preco` < 0), COUNT_IF(`qt_unidade_preco` = 0)
  ) AS (coluna, tipo, preenchidos, minimo, maximo, media, mediana, negativos, zeros)
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras
)
ORDER BY coluna;

## 11. Totais para conciliação com o SAP

**Use esta tabela para bater os totais contra o extrato do SAP.**

Some as mesmas colunas no Excel e compare. Divergência de total detecta registro
faltando ou duplicado — cobre o ponto cego da contagem de linhas.

> ⚠️ **Nesta tabela, `qt_pedido` e `vl_liquido` NÃO podem ser somados direto** — são valores do item que se repetem em cada divisão de remessa. Os totais abaixo estão **inflados**. Use a seção 18.3, que traz a soma deduplicada.

In [0]:
-- TOTAIS PARA CONCILIACAO COM O SAP
SELECT coluna, total_numerico, total_arredondado, linhas_preenchidas
FROM (
  SELECT stack(4,
    'qt_pedido', CAST(SUM(`qt_pedido`) AS DOUBLE), CAST(ROUND(SUM(`qt_pedido`), 2) AS STRING), COUNT(`qt_pedido`),
    'qt_a_fornecer', CAST(SUM(`qt_a_fornecer`) AS DOUBLE), CAST(ROUND(SUM(`qt_a_fornecer`), 2) AS STRING), COUNT(`qt_a_fornecer`),
    'vl_liquido', CAST(SUM(`vl_liquido`) AS DOUBLE), CAST(ROUND(SUM(`vl_liquido`), 2) AS STRING), COUNT(`vl_liquido`),
    'qt_unidade_preco', CAST(SUM(`qt_unidade_preco`) AS DOUBLE), CAST(ROUND(SUM(`qt_unidade_preco`), 2) AS STRING), COUNT(`qt_unidade_preco`)
  ) AS (coluna, total_numerico, total_arredondado, linhas_preenchidas)
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras
)
ORDER BY coluna;

## 12. Datas

Todas as datas desta tabela estão em tipo nativo `date`. A coluna `sentinela_9999`
conta ocorrências de `9999-12-31`, usada pelo SAP para "sem prazo definido".

In [0]:
-- DATAS EM TIPO NATIVO
SELECT coluna, nulos, minimo, maximo, distintas, futuras, sentinela_9999
FROM (
  SELECT stack(3,
    'dt_remessa', COUNT_IF(`dt_remessa` IS NULL), CAST(MIN(`dt_remessa`) AS STRING), CAST(MAX(`dt_remessa`) AS STRING), COUNT(DISTINCT `dt_remessa`), COUNT_IF(`dt_remessa` > current_date()), COUNT_IF(`dt_remessa` = DATE'9999-12-31'),
    'dt_criacao', COUNT_IF(`dt_criacao` IS NULL), CAST(MIN(`dt_criacao`) AS STRING), CAST(MAX(`dt_criacao`) AS STRING), COUNT(DISTINCT `dt_criacao`), COUNT_IF(`dt_criacao` > current_date()), COUNT_IF(`dt_criacao` = DATE'9999-12-31'),
    'dateingest', COUNT_IF(`dateingest` IS NULL), CAST(MIN(`dateingest`) AS STRING), CAST(MAX(`dateingest`) AS STRING), COUNT(DISTINCT `dateingest`), COUNT_IF(`dateingest` > current_date()), COUNT_IF(`dateingest` = DATE'9999-12-31')
  ) AS (coluna, nulos, minimo, maximo, distintas, futuras, sentinela_9999)
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras
)
ORDER BY coluna;

## 13. Códigos — zeros à esquerda e formato

**Armadilha:** o SAP exporta `425263` e o Datalake grava `000000000000425263`.
Sem normalizar, o join dá 0% de match.

A coluna `alertas` resume o que exige tratamento antes da comparação.

In [0]:
-- CODIGOS: ZEROS A ESQUERDA E FORMATO
SELECT coluna, tipo, vazios, len_min, len_max, com_zeros_esq,
       distintos_bruto, distintos_sem_zeros,
       distintos_bruto - distintos_sem_zeros AS colisoes,
       CONCAT_WS(' | ',
         CASE WHEN tipo LIKE 'big%' OR tipo LIKE '%int%'
              THEN 'TIPO NUMERICO - zeros ja perdidos' END,
         CASE WHEN com_zeros_esq > 0 THEN 'normalizar antes do join' END,
         CASE WHEN len_min <> len_max THEN 'comprimento variavel' END,
         CASE WHEN distintos_bruto - distintos_sem_zeros > 0 THEN 'COLISAO ao remover zeros' END
       ) AS alertas
FROM (
  SELECT stack(6,
    'num_pedido_compra', 'string', COUNT_IF(CAST(`num_pedido_compra` AS STRING) IS NULL OR trim(CAST(`num_pedido_compra` AS STRING)) = ''), MIN(length(trim(CAST(`num_pedido_compra` AS STRING)))), MAX(length(trim(CAST(`num_pedido_compra` AS STRING)))), COUNT_IF(trim(CAST(`num_pedido_compra` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`num_pedido_compra` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_pedido_compra` AS STRING)), '^0+', '')),
    'num_item_pedido_compra', 'string', COUNT_IF(CAST(`num_item_pedido_compra` AS STRING) IS NULL OR trim(CAST(`num_item_pedido_compra` AS STRING)) = ''), MIN(length(trim(CAST(`num_item_pedido_compra` AS STRING)))), MAX(length(trim(CAST(`num_item_pedido_compra` AS STRING)))), COUNT_IF(trim(CAST(`num_item_pedido_compra` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`num_item_pedido_compra` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_item_pedido_compra` AS STRING)), '^0+', '')),
    'num_divisao_remessa', 'string', COUNT_IF(CAST(`num_divisao_remessa` AS STRING) IS NULL OR trim(CAST(`num_divisao_remessa` AS STRING)) = ''), MIN(length(trim(CAST(`num_divisao_remessa` AS STRING)))), MAX(length(trim(CAST(`num_divisao_remessa` AS STRING)))), COUNT_IF(trim(CAST(`num_divisao_remessa` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`num_divisao_remessa` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_divisao_remessa` AS STRING)), '^0+', '')),
    'cod_material', 'string', COUNT_IF(CAST(`cod_material` AS STRING) IS NULL OR trim(CAST(`cod_material` AS STRING)) = ''), MIN(length(trim(CAST(`cod_material` AS STRING)))), MAX(length(trim(CAST(`cod_material` AS STRING)))), COUNT_IF(trim(CAST(`cod_material` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_material` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_material` AS STRING)), '^0+', '')),
    'cod_centro', 'string', COUNT_IF(CAST(`cod_centro` AS STRING) IS NULL OR trim(CAST(`cod_centro` AS STRING)) = ''), MIN(length(trim(CAST(`cod_centro` AS STRING)))), MAX(length(trim(CAST(`cod_centro` AS STRING)))), COUNT_IF(trim(CAST(`cod_centro` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_centro` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_centro` AS STRING)), '^0+', '')),
    'cod_fornecedor', 'string', COUNT_IF(CAST(`cod_fornecedor` AS STRING) IS NULL OR trim(CAST(`cod_fornecedor` AS STRING)) = ''), MIN(length(trim(CAST(`cod_fornecedor` AS STRING)))), MAX(length(trim(CAST(`cod_fornecedor` AS STRING)))), COUNT_IF(trim(CAST(`cod_fornecedor` AS STRING)) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(CAST(`cod_fornecedor` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_fornecedor` AS STRING)), '^0+', ''))
  ) AS (coluna, tipo, vazios, len_min, len_max, com_zeros_esq,
        distintos_bruto, distintos_sem_zeros)
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras
)
ORDER BY coluna;

## 14. Checksum de linha

Se `linhas` > `linhas_unicas`, existem registros 100% idênticos — duplicata real,
não granularidade adicional.

In [0]:
-- CHECKSUM DE LINHA
SELECT COUNT(*)                                            AS linhas,
       COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`num_pedido_compra` AS STRING), ''), COALESCE(CAST(`num_item_pedido_compra` AS STRING), ''), COALESCE(CAST(`num_divisao_remessa` AS STRING), ''), COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`cod_centro` AS STRING), ''), COALESCE(CAST(`cod_grupo_compradores` AS STRING), ''), COALESCE(CAST(`nm_grupo_compradores` AS STRING), ''), COALESCE(CAST(`nm_usuario_criacao` AS STRING), ''), COALESCE(CAST(`cod_grupo_mercadoria` AS STRING), ''), COALESCE(CAST(`qt_pedido` AS STRING), ''), COALESCE(CAST(`qt_a_fornecer` AS STRING), ''), COALESCE(CAST(`cod_unidade_medida` AS STRING), ''), COALESCE(CAST(`vl_liquido` AS STRING), ''), COALESCE(CAST(`qt_unidade_preco` AS STRING), ''), COALESCE(CAST(`cod_moeda` AS STRING), ''), COALESCE(CAST(`ind_entrega_completa` AS STRING), ''), COALESCE(CAST(`dt_remessa` AS STRING), ''), COALESCE(CAST(`dt_criacao` AS STRING), ''), COALESCE(CAST(`cod_fornecedor` AS STRING), ''), COALESCE(CAST(`nm_fornecedor` AS STRING), ''), COALESCE(CAST(`cod_incoterms` AS STRING), ''), COALESCE(CAST(`nm_incoterms_local` AS STRING), ''), COALESCE(CAST(`cod_tipo_pedido_compra` AS STRING), ''), COALESCE(CAST(`dateingest` AS STRING), ''))))            AS linhas_unicas,
       COUNT(*) - COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`num_pedido_compra` AS STRING), ''), COALESCE(CAST(`num_item_pedido_compra` AS STRING), ''), COALESCE(CAST(`num_divisao_remessa` AS STRING), ''), COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`cod_centro` AS STRING), ''), COALESCE(CAST(`cod_grupo_compradores` AS STRING), ''), COALESCE(CAST(`nm_grupo_compradores` AS STRING), ''), COALESCE(CAST(`nm_usuario_criacao` AS STRING), ''), COALESCE(CAST(`cod_grupo_mercadoria` AS STRING), ''), COALESCE(CAST(`qt_pedido` AS STRING), ''), COALESCE(CAST(`qt_a_fornecer` AS STRING), ''), COALESCE(CAST(`cod_unidade_medida` AS STRING), ''), COALESCE(CAST(`vl_liquido` AS STRING), ''), COALESCE(CAST(`qt_unidade_preco` AS STRING), ''), COALESCE(CAST(`cod_moeda` AS STRING), ''), COALESCE(CAST(`ind_entrega_completa` AS STRING), ''), COALESCE(CAST(`dt_remessa` AS STRING), ''), COALESCE(CAST(`dt_criacao` AS STRING), ''), COALESCE(CAST(`cod_fornecedor` AS STRING), ''), COALESCE(CAST(`nm_fornecedor` AS STRING), ''), COALESCE(CAST(`cod_incoterms` AS STRING), ''), COALESCE(CAST(`nm_incoterms_local` AS STRING), ''), COALESCE(CAST(`cod_tipo_pedido_compra` AS STRING), ''), COALESCE(CAST(`dateingest` AS STRING), '')))) AS linhas_100pct_identicas,
       CASE WHEN COUNT(*) = COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`num_pedido_compra` AS STRING), ''), COALESCE(CAST(`num_item_pedido_compra` AS STRING), ''), COALESCE(CAST(`num_divisao_remessa` AS STRING), ''), COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`cod_centro` AS STRING), ''), COALESCE(CAST(`cod_grupo_compradores` AS STRING), ''), COALESCE(CAST(`nm_grupo_compradores` AS STRING), ''), COALESCE(CAST(`nm_usuario_criacao` AS STRING), ''), COALESCE(CAST(`cod_grupo_mercadoria` AS STRING), ''), COALESCE(CAST(`qt_pedido` AS STRING), ''), COALESCE(CAST(`qt_a_fornecer` AS STRING), ''), COALESCE(CAST(`cod_unidade_medida` AS STRING), ''), COALESCE(CAST(`vl_liquido` AS STRING), ''), COALESCE(CAST(`qt_unidade_preco` AS STRING), ''), COALESCE(CAST(`cod_moeda` AS STRING), ''), COALESCE(CAST(`ind_entrega_completa` AS STRING), ''), COALESCE(CAST(`dt_remessa` AS STRING), ''), COALESCE(CAST(`dt_criacao` AS STRING), ''), COALESCE(CAST(`cod_fornecedor` AS STRING), ''), COALESCE(CAST(`nm_fornecedor` AS STRING), ''), COALESCE(CAST(`cod_incoterms` AS STRING), ''), COALESCE(CAST(`nm_incoterms_local` AS STRING), ''), COALESCE(CAST(`cod_tipo_pedido_compra` AS STRING), ''), COALESCE(CAST(`dateingest` AS STRING), ''))))
            THEN 'OK - nenhuma linha totalmente identica'
            ELSE 'ATENCAO - existem linhas identicas em todos os campos' END AS veredito
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras;

## 15. Amostra de linhas completas

> 🔒 As colunas `nm_usuario_criacao` aparecem **mascaradas por hash**, conforme instrução de LGPD no schema.

In [0]:
-- AMOSTRA (colunas sob LGPD mascaradas por hash)
SELECT `num_pedido_compra`,
       `num_item_pedido_compra`,
       `num_divisao_remessa`,
       `cod_material`,
       `cod_centro`,
       `cod_grupo_compradores`,
       `nm_grupo_compradores`,
       substring(md5(`nm_usuario_criacao`), 1, 8) AS `nm_usuario_criacao_hash`,  -- LGPD: valor mascarado,
       `cod_grupo_mercadoria`,
       `qt_pedido`,
       `qt_a_fornecer`,
       `cod_unidade_medida`,
       `vl_liquido`,
       `qt_unidade_preco`,
       `cod_moeda`,
       `ind_entrega_completa`,
       `dt_remessa`,
       `dt_criacao`,
       `cod_fornecedor`,
       `nm_fornecedor`,
       `cod_incoterms`,
       `nm_incoterms_local`,
       `cod_tipo_pedido_compra`,
       `dateingest`
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras
ORDER BY `num_pedido_compra`, `num_item_pedido_compra`, `num_divisao_remessa`
LIMIT 20;

## 16. Distribuição interna

Como o volume se reparte nas principais dimensões.

In [0]:
-- 16. DISTRIBUICAO POR cod_centro
SELECT COALESCE(NULLIF(trim(CAST(`cod_centro` AS STRING)), ''), '(vazio)') AS `cod_centro`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras
GROUP BY COALESCE(NULLIF(trim(CAST(`cod_centro` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [0]:
-- 16. DISTRIBUICAO POR cod_grupo_compradores
SELECT COALESCE(NULLIF(trim(CAST(`cod_grupo_compradores` AS STRING)), ''), '(vazio)') AS `cod_grupo_compradores`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras
GROUP BY COALESCE(NULLIF(trim(CAST(`cod_grupo_compradores` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [0]:
-- 16. DISTRIBUICAO POR cod_tipo_pedido_compra
SELECT COALESCE(NULLIF(trim(CAST(`cod_tipo_pedido_compra` AS STRING)), ''), '(vazio)') AS `cod_tipo_pedido_compra`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras
GROUP BY COALESCE(NULLIF(trim(CAST(`cod_tipo_pedido_compra` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

In [0]:
-- 16. DISTRIBUICAO POR cod_grupo_mercadoria
SELECT COALESCE(NULLIF(trim(CAST(`cod_grupo_mercadoria` AS STRING)), ''), '(vazio)') AS `cod_grupo_mercadoria`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras), 2) AS pct
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras
GROUP BY COALESCE(NULLIF(trim(CAST(`cod_grupo_mercadoria` AS STRING)), ''), '(vazio)')
ORDER BY linhas DESC
LIMIT 40;

## 17. Freshness

In [0]:
-- 17. FRESHNESS
SELECT `dateingest` AS data_carga, COUNT(*) AS linhas
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras
GROUP BY `dateingest`
ORDER BY data_carga DESC
LIMIT 30;

In [0]:
-- 17.1 SNAPSHOT OU HISTORICO?
SELECT MIN(`dateingest`)            AS primeira_carga,
       MAX(`dateingest`)            AS ultima_carga,
       COUNT(DISTINCT `dateingest`) AS cargas_distintas,
       CASE WHEN COUNT(DISTINCT `dateingest`) = 1
            THEN 'SNAPSHOT - substitui a cada carga; chave NAO precisa da data'
            ELSE 'HISTORICO - acumula; a chave DEVE incluir a data de carga' END AS veredito
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras;

## 18. Análises específicas — ZGLMM147

### 18.1 Divisões de remessa por item

O grão da tabela é a **divisão de remessa**, não o item. O comentário registra que
`num_divisao_remessa` é NULA quando o item não tem programação — eram 10 itens
no universo em 2026-09-10.

In [0]:
-- 18.1 DIVISOES POR ITEM
SELECT divisoes_no_item, COUNT(*) AS itens
FROM (SELECT num_pedido_compra, num_item_pedido_compra,
             COUNT(*) AS divisoes_no_item
        FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras
       GROUP BY num_pedido_compra, num_item_pedido_compra)
GROUP BY divisoes_no_item
ORDER BY divisoes_no_item
LIMIT 40;

In [0]:
-- 18.1b ITENS SEM DIVISAO DE REMESSA
-- O comentario registra 10 itens nessa condicao em 2026-09-10.
SELECT COUNT(*)                                                       AS linhas_sem_divisao,
       COUNT_IF(dt_remessa IS NULL)                                   AS tambem_sem_dt_remessa,
       COUNT_IF(qt_a_fornecer IS NULL)                                AS tambem_sem_qt_a_fornecer,
       CASE WHEN COUNT(*) = COUNT_IF(dt_remessa IS NULL AND qt_a_fornecer IS NULL)
            THEN 'CONFIRMADO - as tres colunas sao nulas juntas'
            ELSE 'DIVERGE do comportamento declarado' END             AS veredito
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras
WHERE num_divisao_remessa IS NULL OR trim(num_divisao_remessa) = '';

### 18.2 Domínio de `qt_a_fornecer` — saldo negativo é esperado

O comentário registra a medição de 2026-09-09 sobre 7.513.595 divisões:

| Faixa | % declarado |
|---|---|
| Saldo zero (recebido integralmente) | 68,4% |
| Saldo positivo | 22,0% |
| **Saldo negativo** (over-delivery) | **9,6%** |

Saldo negativo é recebimento acima do programado, previsto pela tolerância de excesso
do item — **não é erro de carga**.

In [0]:
-- 18.2 DOMINIO DE qt_a_fornecer
SELECT COUNT(*)                                           AS divisoes,
       COUNT_IF(qt_a_fornecer = 0)                        AS saldo_zero,
       ROUND(100.0 * COUNT_IF(qt_a_fornecer = 0) / COUNT(*), 2)  AS pct_zero,
       COUNT_IF(qt_a_fornecer > 0)                        AS saldo_positivo,
       ROUND(100.0 * COUNT_IF(qt_a_fornecer > 0) / COUNT(*), 2)  AS pct_positivo,
       COUNT_IF(qt_a_fornecer < 0)                        AS saldo_negativo,
       ROUND(100.0 * COUNT_IF(qt_a_fornecer < 0) / COUNT(*), 2)  AS pct_negativo,
       COUNT_IF(qt_a_fornecer IS NULL)                    AS sem_divisao,
       ROUND(MIN(qt_a_fornecer), 3)                       AS menor_saldo,
       ROUND(MAX(qt_a_fornecer), 3)                       AS maior_saldo
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras;

### 18.3 Totais COM e SEM deduplicação

**Ponto crítico da conciliação.** `qt_pedido` e `vl_liquido` são valores do **item**
e se repetem em todas as divisões. Somar direto **infla o total**.

Compare as duas colunas abaixo: a diferença é o tamanho do erro que uma soma ingênua
produziria. Use sempre a coluna **deduplicada** para bater contra o SAP.

In [0]:
-- 18.3 TOTAIS: SOMA INGENUA vs DEDUPLICADA
WITH dedup AS (
  SELECT num_pedido_compra, num_item_pedido_compra,
         MAX(qt_pedido)  AS qt_pedido,
         MAX(vl_liquido) AS vl_liquido
  FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras
  GROUP BY num_pedido_compra, num_item_pedido_compra
)
SELECT 'qt_pedido' AS coluna,
       (SELECT ROUND(SUM(qt_pedido), 3) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras)   AS soma_ingenua_INFLADA,
       (SELECT ROUND(SUM(qt_pedido), 3) FROM dedup) AS soma_correta_dedup,
       (SELECT ROUND(SUM(qt_pedido), 3) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras)
     - (SELECT ROUND(SUM(qt_pedido), 3) FROM dedup) AS diferenca
UNION ALL
SELECT 'vl_liquido',
       (SELECT ROUND(SUM(vl_liquido), 2) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras),
       (SELECT ROUND(SUM(vl_liquido), 2) FROM dedup),
       (SELECT ROUND(SUM(vl_liquido), 2) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras)
     - (SELECT ROUND(SUM(vl_liquido), 2) FROM dedup);

### 18.4 `ind_entrega_completa` — STRING, não boolean

O comentário registra o domínio medido em 2026-09-09: `X` (3.343.956) e vazio (928.127).
O prefixo `ind_` não significa boolean nesta plataforma.

In [0]:
-- 18.4 DOMINIO DE ind_entrega_completa
SELECT COALESCE(NULLIF(trim(ind_entrega_completa), ''), '(vazio)') AS ind_entrega_completa,
       COUNT(*)                                     AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras), 2) AS pct,
       COUNT_IF(qt_a_fornecer > 0)                  AS com_saldo_a_fornecer,
       CASE WHEN trim(COALESCE(ind_entrega_completa,'')) NOT IN ('X', '')
            THEN 'FORA DO DOMINIO DECLARADO' ELSE 'dominio esperado' END AS veredito
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras
GROUP BY COALESCE(NULLIF(trim(ind_entrega_completa), ''), '(vazio)'),
         trim(COALESCE(ind_entrega_completa,''))
ORDER BY linhas DESC;

### 18.5 Escopo: pedidos de transferência não existem aqui

O comentário registra que os tipos de transferência (UB, ZUB, ZRE e afins)
**não aparecem** nesta tabela — o universo exclui pedidos com centro de saída
de estoque (RESWK) preenchido, igual à transação SAP.

**Ao extrair do SAP, aplique o mesmo filtro** — senão a comparação acusará
falsos registros ausentes.

In [0]:
-- 18.5 TIPOS DE PEDIDO PRESENTES
SELECT COALESCE(NULLIF(trim(cod_tipo_pedido_compra), ''), '(vazio)') AS cod_tipo_pedido_compra,
       COUNT(*)                                  AS linhas,
       COUNT(DISTINCT num_pedido_compra)         AS pedidos,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras), 2) AS pct,
       CASE WHEN trim(COALESCE(cod_tipo_pedido_compra,'')) IN ('UB','ZUB','ZRE')
            THEN 'TIPO DE TRANSFERENCIA - nao deveria existir aqui'
            ELSE 'ok' END AS veredito
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras
GROUP BY COALESCE(NULLIF(trim(cod_tipo_pedido_compra), ''), '(vazio)'),
         trim(COALESCE(cod_tipo_pedido_compra,''))
ORDER BY linhas DESC
LIMIT 30;

### 18.6 Sentinela 9999-12-31 em `dt_remessa`

O comentário declara que `9999-12-31` é **válido** (item sem prazo definido)
e que `00000000` virou NULL.

In [0]:
-- 18.6 SENTINELA DE DATA
SELECT COUNT(*)                                        AS divisoes,
       COUNT_IF(dt_remessa = DATE'9999-12-31')         AS sem_prazo_definido,
       ROUND(100.0 * COUNT_IF(dt_remessa = DATE'9999-12-31') / COUNT(*), 4) AS pct_sem_prazo,
       COUNT_IF(dt_remessa IS NULL)                    AS dt_remessa_nula,
       MIN(CASE WHEN dt_remessa <> DATE'9999-12-31' THEN dt_remessa END) AS remessa_mais_antiga,
       MAX(CASE WHEN dt_remessa <> DATE'9999-12-31' THEN dt_remessa END) AS remessa_mais_recente,
       COUNT_IF(dt_remessa < dt_criacao)               AS remessa_antes_da_criacao
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras;

### 18.7 Cobertura do cadastro de fornecedor

O comentário declara cobertura de **96,68%** — o join é LEFT e o nome pode vir nulo
legitimamente.

In [0]:
-- 18.7 COBERTURA DE nm_fornecedor
SELECT COUNT(*)                                                      AS linhas,
       COUNT_IF(trim(COALESCE(nm_fornecedor,'')) <> '')              AS com_nome,
       COUNT_IF(trim(COALESCE(nm_fornecedor,'')) = '')               AS sem_nome,
       ROUND(100.0 * COUNT_IF(trim(COALESCE(nm_fornecedor,'')) <> '') / COUNT(*), 2) AS pct_cobertura,
       COUNT(DISTINCT cod_fornecedor)                                AS fornecedores_distintos,
       COUNT(DISTINCT CASE WHEN trim(COALESCE(nm_fornecedor,'')) = ''
                           THEN cod_fornecedor END)                  AS fornecedores_sem_cadastro
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras;

### 18.8 🔒 Usuário de criação — análise agregada (LGPD)

O comentário do schema instrui: *"tratar como sensível, validar por contagem ou agregado,
**nunca amostrar valor literal** nem usar como literal em WHERE"*.

Esta célula traz **apenas contagens** — nenhum valor literal é exibido.

In [0]:
-- 18.8 USUARIO DE CRIACAO (LGPD - somente agregados)
SELECT COUNT(*)                                                  AS linhas,
       COUNT(DISTINCT nm_usuario_criacao)                        AS usuarios_distintos,
       COUNT_IF(trim(COALESCE(nm_usuario_criacao,'')) = '')      AS sem_usuario,
       ROUND(COUNT(*) / NULLIF(COUNT(DISTINCT nm_usuario_criacao), 0), 1) AS media_linhas_por_usuario
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras;

In [0]:
-- 18.8b DISTRIBUICAO POR USUARIO (identificador mascarado por hash)
SELECT substring(md5(nm_usuario_criacao), 1, 8) AS usuario_hash,
       COUNT(*)                                 AS linhas,
       COUNT(DISTINCT num_pedido_compra)        AS pedidos
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras
WHERE trim(COALESCE(nm_usuario_criacao,'')) <> ''
GROUP BY substring(md5(nm_usuario_criacao), 1, 8)
ORDER BY linhas DESC
LIMIT 20;

## 19. Amostra ampliada

O notebook exploratório não extrai a base completa. Para isso, use um notebook de cenário
com o recorte escolhido na seção 3.

In [0]:
-- 19. AMOSTRA AMPLIADA (colunas sob LGPD mascaradas)
SELECT `num_pedido_compra`,
       `num_item_pedido_compra`,
       `num_divisao_remessa`,
       `cod_material`,
       `cod_centro`,
       `cod_grupo_compradores`,
       `nm_grupo_compradores`,
       substring(md5(`nm_usuario_criacao`), 1, 8) AS `nm_usuario_criacao_hash`,
       `cod_grupo_mercadoria`,
       `qt_pedido`,
       `qt_a_fornecer`,
       `cod_unidade_medida`,
       `vl_liquido`,
       `qt_unidade_preco`,
       `cod_moeda`,
       `ind_entrega_completa`,
       `dt_remessa`,
       `dt_criacao`,
       `cod_fornecedor`,
       `nm_fornecedor`,
       `cod_incoterms`,
       `nm_incoterms_local`,
       `cod_tipo_pedido_compra`,
       `dateingest`
FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras
ORDER BY `num_pedido_compra`, `num_item_pedido_compra`, `num_divisao_remessa`
LIMIT 500;

## 20. Resumo

**Copie esta saída** e envie ao agente junto com o notebook.

In [0]:
-- 20. RESUMO
WITH t AS (SELECT COUNT(*) AS total FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras),
g AS (
  SELECT 'num_pedido_compra' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `num_pedido_compra` FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras)
UNION ALL
  SELECT 'num_pedido_compra + num_item_pedido_compra' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `num_pedido_compra`, `num_item_pedido_compra` FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras)
UNION ALL
  SELECT 'num_pedido_compra + num_item_pedido_compra + num_divisao_remessa' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `num_pedido_compra`, `num_item_pedido_compra`, `num_divisao_remessa` FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm147_followup_compras)
)
SELECT 'IDENTIFICACAO' AS bloco, 'transacao' AS item, 'ZGLMM147' AS valor
UNION ALL SELECT 'IDENTIFICACAO', 'tabela', 'tbl_ds_pro_zglmm147_followup_compras'
UNION ALL SELECT 'IDENTIFICACAO', 'linhas', format_number((SELECT total FROM t), 0)
UNION ALL SELECT 'IDENTIFICACAO', 'colunas', '24'
UNION ALL SELECT 'IDENTIFICACAO', 'clustering declarado',
       'num_pedido_compra, cod_material, cod_centro, dt_criacao'
UNION ALL SELECT 'IDENTIFICACAO', 'coluna de recorte', 'cod_centro'
UNION ALL
SELECT 'GRANULARIDADE', g.chave,
       CONCAT(format_number(g.d, 0), ' distintos | ',
              CAST(ROUND(t.total / g.d, 4) AS STRING), ' linhas/chave | ',
              CASE WHEN g.d = t.total THEN 'CHAVE UNICA' ELSE 'nao unica' END)
  FROM g CROSS JOIN t
UNION ALL
SELECT 'CHAVE REAL', 'sugerida',
       COALESCE((SELECT MIN(g.chave) FROM g CROSS JOIN t WHERE g.d = t.total),
                'NENHUMA - investigar')
ORDER BY bloco, item;

---

## Próximo passo

1. Escolher o recorte de teste com base na **seção 3**.
2. Gerar o notebook de cenário com esse recorte.
3. Extrair a ZGLMM147 no SAP com o mesmo recorte e na **mesma data** do snapshot.
4. Enviar ao agente de validação: este notebook executado + os arquivos do SAP.

### Checklist antes de comparar

- [ ] Chave real identificada (seção 4)
- [ ] Colunas 100% nulas conferidas no SAP antes de classificar como erro (seção 6)
- [ ] Zeros à esquerda normalizados nos dois lados (seção 13)
- [ ] Tolerância de 0,005 aplicada em campos `double` (seção 10)
- [ ] Totais numéricos conferidos (seção 11)
- [ ] 🔒 Colunas sob LGPD tratadas apenas por agregado, nunca por valor literal
- [ ] ⚠️ `qt_pedido` e `vl_liquido` deduplicados por pedido+item antes de somar
- [ ] Filtro de transferência (RESWK) aplicado também na extração do SAP
- [ ] Alertas de partida verificados (seção 18)
